In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
pd.set_option('display.max_colwidth', None)

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MAGAZINE_1 = 'scopus'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [4]:
from bertopic import BERTopic

model_path = cfg.MODELS_FOLDER / f'{MAGAZINE_1}/model_0.343.safetensors'
model_1 = BERTopic.load(model_path, 
                      embedding_model=cfg.EMBEDDING_MODEL
                      )

In [5]:
model_name ="Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.15it/s]


In [12]:
def evaluate_topic_assignment(keywords_list):
    messages = [
        {
            "role": "user",
            "content": (
                "Create a short topic label from the keywords below.\n"
                "Return ONLY the label as a short noun phrase (3–6 words).\n"
                
                "Keywords:\n"
                f"{'\n'.join(f'- {key}' for key in keywords_list)}\n\n"
                #"Topic label:"
            )
        }
    ]

    print(messages)


    testo = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([testo],
                              return_tensors="pt",
                              truncation=True,
                              max_length=100).to(model.device)
    with torch.inference_mode():
        generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=15,
        do_sample = False,
        temperature = 0.0,
        repetition_penalty=1.1
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

    # parsing thinking content
    try:
    # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    return tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

In [14]:
topic_names = {}
for i in range(0,len(set(model_1.topics_))-1):
    topic_keywords = [ keywords_pair[0] for keywords_pair in model_1.get_topic(i) ]
    topic_name = evaluate_topic_assignment(topic_keywords)
    topic_names[i] = topic_name
    

[{'role': 'user', 'content': 'Create a short topic label from the keywords below.\nReturn ONLY the label as a short noun phrase (3–6 words).\nKeywords:\n- electrochemical biosensor\n- base biosensor\n- biosensing\n- detection limit\n- aptasensor\n- immunosensor\n- fluorescent protein\n- biosensor\n- high sensitivity\n- bacterial cell\n\n'}]
[{'role': 'user', 'content': 'Create a short topic label from the keywords below.\nReturn ONLY the label as a short noun phrase (3–6 words).\nKeywords:\n- resistant tuberculosis\n- tuberculosis drug\n- tuberculosis strain\n- tuberculosis case\n- tuberculosis patient\n- mdr tuberculosis\n- tuberculosis isolate\n- tuberculosis treatment\n- tuberculosis incidence\n- tuberculosis control\n\n'}]
[{'role': 'user', 'content': 'Create a short topic label from the keywords below.\nReturn ONLY the label as a short noun phrase (3–6 words).\nKeywords:\n- wildlife health\n- disease surveillance\n- wildlife disease\n- zoonotic disease\n- zoonotic infection\n- zoo

In [19]:
topic_names = { key:val.strip().replace('\n','') for key,val in topic_names.items() }

In [21]:
model_1.set_topic_labels(topic_names)

In [17]:
model_1.save(
    model_path,
    serialization="safetensors",
    save_ctfidf=True,
    save_embedding_model=False
)

2026-01-27 17:06:09,895 - BERTopic - WARNING: You are saving a BERTopic model without explicitly defining an embedding model.If you are using a sentence-transformers model or a HuggingFace model supportedby sentence-transformers, please save the model by using a pointer towards that model.For example, `save_embedding_model='sentence-transformers/all-mpnet-base-v2'`
